In [55]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [ ]:
df = pd.read_csv("training_data.csv")

# Check
Missing precentages and number of unique values for each feature to decide which are actually categorical feature.

Compute survival target features

In [56]:
missing_percentages = df.isnull().sum() / len(df) * 100
missing_above_20_percent = missing_percentages[missing_percentages > 20]
print(missing_above_20_percent)

CONTRACT_INSTALMENT_AMOUNT         82.017067
CONTRACT_INTEREST_PERIOD           23.167138
CONTRACT_LOAN_TO_VALUE_RATIO       73.406819
CONTRACT_MARKET_VALUE              67.062298
CONTRACT_MORTGAGE_LENDING_VALUE    66.444174
CONTRACT_MORTGAGE_TYPE             66.444174
BORROWER_TYPE_OF_SETTLEMENT        32.982250
TARGET_EVENT_DAY                   96.606526
dtype: float64


In [46]:
for c in df.columns:
    print(c, df[c].nunique())

CONTRACT_ID 1274533
BORROWER_ID 1149851
CONTRACT_BANK_ID 136
CONTRACT_CREDIT_INTERMEDIARY 4
CONTRACT_CREDIT_LOSS 165007
CONTRACT_CURRENCY 3
CONTRACT_DATE_OF_LOAN_AGREEMENT 1072
CONTRACT_DEPT_SERVICE_TO_INCOME 11367
CONTRACT_FREQUENCY_TYPE 8
CONTRACT_INCOME 410880
CONTRACT_INSTALMENT_AMOUNT 61334
CONTRACT_INSTALMENT_AMOUNT_2 158047
CONTRACT_INTEREST_PERIOD 1038
CONTRACT_INTEREST_RATE 4322
CONTRACT_LGD 879303
CONTRACT_LOAN_AMOUNT 835480
CONTRACT_LOAN_CONTRACT_TYPE 8
CONTRACT_LOAN_TO_VALUE_RATIO 11662
CONTRACT_LOAN_TYPE 18
CONTRACT_MARKET_VALUE 274389
CONTRACT_MATURITY_DATE 10744
CONTRACT_MORTGAGE_LENDING_VALUE 271960
CONTRACT_MORTGAGE_TYPE 18
CONTRACT_REFINANCED 4
CONTRACT_RISK_WEIGHTED_ASSETS 23085
CONTRACT_TYPE_OF_INTEREST_REPAYMENT 9
BORROWER_BIRTH_YEAR 101
BORROWER_CITIZENSHIP 41
BORROWER_COUNTRY 53
BORROWER_COUNTY 199
BORROWER_TYPE_OF_CUSTOMER 2
BORROWER_TYPE_OF_SETTLEMENT 8
TARGET_EVENT 3
TARGET_EVENT_DAY 928


## Converting event and Computing duration

event ∈ {0, 1, 2}

0 -> censored (no event)

1 -> default (TARGET_EVENT == 'K')

2 -> prepayment (TARGET_EVENT == 'E')

In [57]:
df["event"] = 0
df.loc[df["TARGET_EVENT"] == "K", "event"] = 1
df.loc[df["TARGET_EVENT"] == "E", "event"] = 2

In [58]:
df['event'].value_counts()

,count
event,
0,1548364
2,43515
1,10874


In [59]:
CENSOR_DAY = max(df["TARGET_EVENT_DAY"].max(), df["CONTRACT_DATE_OF_LOAN_AGREEMENT"].max())

df["event_time"] = np.where(
    df["event"] > 0,
    df["TARGET_EVENT_DAY"],
    CENSOR_DAY
)
df["time"] = df["event_time"] - df["CONTRACT_DATE_OF_LOAN_AGREEMENT"]

df = df[df["time"] > 0].copy()

## Decide Categories

In [60]:
time_col = "time"
event_col = "event"
identifiers = ['CONTRACT_ID', 'BORROWER_ID']
num_but_cat_cols = ['CONTRACT_CREDIT_INTERMEDIARY',
                    'CONTRACT_CURRENCY',
                    'CONTRACT_LOAN_CONTRACT_TYPE',
                    'CONTRACT_REFINANCED',
                    'CONTRACT_TYPE_OF_INTEREST_REPAYMENT',
                    'BORROWER_CITIZENSHIP',
                    'BORROWER_COUNTRY',
                    'BORROWER_COUNTY',
                    'BORROWER_TYPE_OF_SETTLEMENT',
               ]

cat_features = ['CONTRACT_CREDIT_INTERMEDIARY',
                'CONTRACT_CURRENCY',
                'CONTRACT_LOAN_CONTRACT_TYPE',
                'CONTRACT_REFINANCED',
                'CONTRACT_TYPE_OF_INTEREST_REPAYMENT',
                'CONTRACT_FREQUENCY_TYPE', #
                'CONTRACT_LOAN_TYPE', #
                #'CONTRACT_MORTGAGE_TYPE', 66.4% missing
                'BORROWER_CITIZENSHIP', #
                'BORROWER_COUNTRY', #
                'BORROWER_COUNTY', #
                'BORROWER_TYPE_OF_SETTLEMENT', #
                'BORROWER_TYPE_OF_CUSTOMER', #
                ]

num_features = ['CONTRACT_CREDIT_LOSS', # leakage candidate
                'CONTRACT_DEPT_SERVICE_TO_INCOME',
                'CONTRACT_INTEREST_PERIOD',
                'CONTRACT_INTEREST_RATE',
                'CONTRACT_LOAN_AMOUNT',
                'CONTRACT_LGD', # leakage candidate
                #'CONTRACT_INSTALMENT_AMOUNT', 82% missing
                'CONTRACT_INSTALMENT_AMOUNT_2',
                'CONTRACT_INCOME',
                #'CONTRACT_LOAN_TO_VALUE_RATIO', 73.4% missing
                #'CONTRACT_MARKET_VALUE', 67.1% missing
                #'CONTRACT_MORTGAGE_LENDING_VALUE', 66.4% missing
                'CONTRACT_MATURITY_DATE',
                'CONTRACT_RISK_WEIGHTED_ASSETS', # leakage candidate
                'BORROWER_BIRTH_YEAR',
                ]
keep_cols = (
    [time_col, event_col]
    + identifiers
    + num_features
    + cat_features
)
keep_cols = list(dict.fromkeys(keep_cols))
df = df.loc[:, [c for c in keep_cols if c in df.columns]].copy()

# Contract_id de-duplication

In [61]:
def fast_mode(s):
    s = s.dropna()
    if s.empty:
        return np.nan
    return s.value_counts().index[0]

In [62]:
CONTRACT_ID = "CONTRACT_ID"
BORROWER_ID = "BORROWER_ID"
time_col = "time"
event_col = "event"

g = df.groupby(CONTRACT_ID, sort=False)

# --- 1) contract targets: event (assumed consistent) + time (typical conditional min/max) ---
event_first = g[event_col].first().rename(event_col)

t_min = g[time_col].min().rename("t_min")
t_max = g[time_col].max().rename("t_max")

targets = pd.concat([event_first, t_min, t_max], axis=1).reset_index()
targets[time_col] = np.where(targets[event_col] > 0, targets["t_min"], targets["t_max"])
targets = targets[[CONTRACT_ID, event_col, time_col]]


In [63]:
# --- 2) borrower engineered features ---
borrow_agg = g[BORROWER_ID].agg(
    n_borrowers="nunique",
    borrower_id_mode=fast_mode,
).reset_index()

In [67]:
# --- 3) numeric aggregations ---
# a) real numeric -> mean by default, but:
#    - CONTRACT_MATURITY_DATE -> max (as you required)
# b) "num_but_cat_cols" -> treat as categorical -> MODE (not mean)

num_cols_present = [c for c in num_features if c in df.columns]
nbc_present = [c for c in num_but_cat_cols if c in df.columns]

maturity_col = "CONTRACT_MATURITY_DATE"
birth_year_col = "BORROWER_BIRTH_YEAR"

num_mean_cols = [c for c in num_cols_present if c not in [maturity_col, birth_year_col]]

num_mean = (
    g[num_mean_cols].mean(numeric_only=True).reset_index()
    if len(num_mean_cols) > 0 else pd.DataFrame({CONTRACT_ID: targets[CONTRACT_ID]})
)

In [68]:
# median for birth year (if present)
if birth_year_col in df.columns and birth_year_col in num_cols_present:
    birth_median = g[birth_year_col].median().rename(birth_year_col).reset_index()
else:
    birth_median = pd.DataFrame({CONTRACT_ID: targets[CONTRACT_ID]})

# max for maturity date (if present)
if maturity_col in df.columns and maturity_col in num_cols_present:
    maturity_max = g[maturity_col].max().rename(maturity_col).reset_index()
else:
    maturity_max = pd.DataFrame({CONTRACT_ID: targets[CONTRACT_ID]})

# "num but cat": take mode (categorical-like)
nbc_mode = (
    g[nbc_present].first().reset_index()
    if len(nbc_present) > 0 else pd.DataFrame({CONTRACT_ID: targets[CONTRACT_ID]})
)

In [70]:
# --- 4) categorical aggregations ---
# cat_features (plus anything not already handled) -> MODE
cat_present = [c for c in cat_features if c in df.columns and c not in nbc_present]
cat_mode = (
    g[cat_present].agg(fast_mode).reset_index()
    if len(cat_present) > 0 else pd.DataFrame({CONTRACT_ID: targets[CONTRACT_ID]})
)

In [71]:
# --- 5) merge all -> unique CONTRACT_ID ---
contract_df = (
    targets
    .merge(borrow_agg, on=CONTRACT_ID, how="left")
    .merge(num_mean, on=CONTRACT_ID, how="left")
    .merge(birth_median, on=CONTRACT_ID, how="left")
    .merge(maturity_max, on=CONTRACT_ID, how="left")
    .merge(nbc_mode, on=CONTRACT_ID, how="left")
    .merge(cat_mode, on=CONTRACT_ID, how="left")
)

In [72]:
# final sanity
contract_df = contract_df[contract_df[time_col] > 0].copy()
print("contract_df shape:", contract_df.shape)
print("duplicate CONTRACT_ID:", contract_df[CONTRACT_ID].duplicated().any())
print(contract_df[event_col].value_counts(dropna=False))
print("n_borrowers distribution (head):")
print(contract_df["n_borrowers"].value_counts().head(10))

contract_df shape: (1274395, 28)
duplicate CONTRACT_ID: False
event
0    1230659
2      33772
1       9964
Name: count, dtype: int64
n_borrowers distribution (head):
n_borrowers
1     1007827
2      219802
3       35366
4        8976
5        1729
6         519
7         119
8          31
9          13
10          6
Name: count, dtype: int64


# Fill missing values

## Check

In [74]:
contract_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1274395 entries, 0 to 1274394
Data columns (total 28 columns):
 #   Column                               Non-Null Count    Dtype  
---  ------                               --------------    -----  
 0   CONTRACT_ID                          1274395 non-null  object 
 1   event                                1274395 non-null  int64  
 2   time                                 1274395 non-null  float64
 3   n_borrowers                          1274395 non-null  int64  
 4   borrower_id_mode                     1274395 non-null  object 
 5   CONTRACT_CREDIT_LOSS                 1242861 non-null  float64
 6   CONTRACT_DEPT_SERVICE_TO_INCOME      1084309 non-null  float64
 7   CONTRACT_INTEREST_PERIOD             928355 non-null   float64
 8   CONTRACT_INTEREST_RATE               1243587 non-null  float64
 9   CONTRACT_LOAN_AMOUNT                 1274395 non-null  float64
 10  CONTRACT_LGD                         1244904 non-null  float64
 11

In [75]:
missing_cols = contract_df.columns[contract_df.isnull().any()].tolist()
print("Columns with missing values:")
for col in missing_cols:
    print(col)

Columns with missing values:
CONTRACT_CREDIT_LOSS
CONTRACT_DEPT_SERVICE_TO_INCOME
CONTRACT_INTEREST_PERIOD
CONTRACT_INTEREST_RATE
CONTRACT_LGD
CONTRACT_INSTALMENT_AMOUNT_2
CONTRACT_INCOME
CONTRACT_RISK_WEIGHTED_ASSETS
BORROWER_BIRTH_YEAR
CONTRACT_CREDIT_INTERMEDIARY
CONTRACT_REFINANCED
CONTRACT_TYPE_OF_INTEREST_REPAYMENT
BORROWER_CITIZENSHIP
BORROWER_COUNTRY
BORROWER_COUNTY
BORROWER_TYPE_OF_SETTLEMENT


In [81]:
contract_df[['CONTRACT_CREDIT_LOSS',
"CONTRACT_DEPT_SERVICE_TO_INCOME",
"CONTRACT_INTEREST_PERIOD",
"CONTRACT_INTEREST_RATE",
"CONTRACT_LGD",
"CONTRACT_INSTALMENT_AMOUNT_2",
"CONTRACT_INCOME",
"CONTRACT_RISK_WEIGHTED_ASSETS",
"BORROWER_BIRTH_YEAR",
"CONTRACT_CREDIT_INTERMEDIARY",
"CONTRACT_REFINANCED",
"CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
"BORROWER_CITIZENSHIP",
"BORROWER_COUNTRY",
"BORROWER_COUNTY",
"BORROWER_TYPE_OF_SETTLEMENT"]].nunique()

,0
CONTRACT_CREDIT_LOSS,164979
CONTRACT_DEPT_SERVICE_TO_INCOME,12402
CONTRACT_INTEREST_PERIOD,1037
CONTRACT_INTEREST_RATE,4752
CONTRACT_LGD,879201
CONTRACT_INSTALMENT_AMOUNT_2,158029
CONTRACT_INCOME,410875
CONTRACT_RISK_WEIGHTED_ASSETS,24045
BORROWER_BIRTH_YEAR,101
CONTRACT_CREDIT_INTERMEDIARY,4


In [82]:
cols_to_check = [
    "CONTRACT_CREDIT_LOSS",
    "CONTRACT_DEPT_SERVICE_TO_INCOME",
    "CONTRACT_INTEREST_PERIOD",
    "CONTRACT_INTEREST_RATE",
    "CONTRACT_LGD",
    "CONTRACT_INSTALMENT_AMOUNT_2",
    "CONTRACT_INCOME",
    "CONTRACT_RISK_WEIGHTED_ASSETS",
    "BORROWER_BIRTH_YEAR",
    "CONTRACT_CREDIT_INTERMEDIARY",
    "CONTRACT_REFINANCED",
    "CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
    "BORROWER_CITIZENSHIP",
    "BORROWER_COUNTRY",
    "BORROWER_COUNTY",
    "BORROWER_TYPE_OF_SETTLEMENT",
]

missing_pct = (
    contract_df[cols_to_check]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

missing_pct

,0
BORROWER_TYPE_OF_SETTLEMENT,38.03
CONTRACT_INTEREST_PERIOD,27.15
CONTRACT_TYPE_OF_INTEREST_REPAYMENT,16.95
CONTRACT_DEPT_SERVICE_TO_INCOME,14.92
CONTRACT_INCOME,14.78
BORROWER_COUNTY,6.96
CONTRACT_CREDIT_INTERMEDIARY,4.23
CONTRACT_REFINANCED,4.23
BORROWER_COUNTRY,2.98
BORROWER_BIRTH_YEAR,2.98


In [86]:
id_cols = ["CONTRACT_ID", "borrower_id_mode"]
target_cols = ["event", "time"]

cat_cols_missing = [
    "BORROWER_COUNTRY",
    "BORROWER_COUNTY",
    "BORROWER_CITIZENSHIP",
    "BORROWER_TYPE_OF_SETTLEMENT",
]
cat_cols_mode = [
    "BORROWER_BIRTH_YEAR",
    "CONTRACT_CREDIT_INTERMEDIARY",
    "CONTRACT_REFINANCED",
    "CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
]

numeric_cols = [
    "CONTRACT_CREDIT_LOSS",
    "CONTRACT_DEPT_SERVICE_TO_INCOME",
    "CONTRACT_INTEREST_PERIOD",
    "CONTRACT_INSTALMENT_AMOUNT_2",
    "CONTRACT_LGD",
    "CONTRACT_INCOME",
    "CONTRACT_INTEREST_RATE",
    "CONTRACT_RISK_WEIGHTED_ASSETS",
]

## Fill
 - with `missing` if it is true categorical column and more than a given precentage missing
 - with mode value if categorical/nominal and not much missing
 - with mean if its numerical with continues values

In [87]:
# 1) (object) → "missing"
for c in cat_cols_missing:
    if c in contract_df.columns:
        contract_df[c] = contract_df[c].fillna("missing")

# 2) numeric-but-categorical → mode
for c in cat_cols_mode:
    mode_val = contract_df[c].mode(dropna=True)
    if not mode_val.empty:
        contract_df[c] = contract_df[c].fillna(mode_val.iloc[0])

# 3) true numeric → mean
for c in numeric_cols:
    contract_df[c] = contract_df[c].fillna(contract_df[c].mean())

# Group rare categories to reduce dimensionality

Plus dtype conversion

In [134]:
def group_rare(series, min_freq=0.01):
    freq = series.value_counts(normalize=True)
    keep = freq[freq >= min_freq].index
    return series.where(series.isin(keep), other="other")

for col in ["BORROWER_COUNTRY", "BORROWER_COUNTY", "BORROWER_CITIZENSHIP"]:
    if col in contract_df.columns:
        contract_df[col] = group_rare(contract_df[col], min_freq=0.01)

In [135]:
contract_df[["BORROWER_COUNTRY", "BORROWER_COUNTY", "BORROWER_CITIZENSHIP"]].nunique()

,0
BORROWER_COUNTRY,3
BORROWER_COUNTY,19
BORROWER_CITIZENSHIP,4


In [139]:
# -------------------------
# DEFINE COLUMN GROUPS
# -------------------------
categorical_cols = [
    "CONTRACT_CREDIT_INTERMEDIARY",
    "CONTRACT_CURRENCY",
    "CONTRACT_LOAN_CONTRACT_TYPE",
    "CONTRACT_REFINANCED",
    "CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
    "BORROWER_CITIZENSHIP",
    "BORROWER_COUNTRY",
    "BORROWER_COUNTY",
    "BORROWER_TYPE_OF_SETTLEMENT",
    "CONTRACT_FREQUENCY_TYPE",
    "CONTRACT_LOAN_TYPE",
    "BORROWER_TYPE_OF_CUSTOMER",
]

numeric_cols = [
    "n_borrowers",
    "CONTRACT_CREDIT_LOSS",
    "CONTRACT_DEPT_SERVICE_TO_INCOME",
    "CONTRACT_INTEREST_PERIOD",
    "CONTRACT_INTEREST_RATE",
    "CONTRACT_LOAN_AMOUNT",
    "CONTRACT_LGD",
    "CONTRACT_INSTALMENT_AMOUNT_2",
    "CONTRACT_INCOME",
    "CONTRACT_RISK_WEIGHTED_ASSETS",
    "BORROWER_BIRTH_YEAR",
    "CONTRACT_MATURITY_DATE",
]

# -------------------------
# APPLY TYPE CONVERSIONS
# -------------------------

# categorical → category dtype
for c in categorical_cols:
    if c in contract_df.columns:
        contract_df[c] = contract_df[c].astype("category")

# numeric → numeric (safety cast)
for c in numeric_cols:
    if c in contract_df.columns:
        contract_df[c] = pd.to_numeric(contract_df[c], errors="coerce")

# Split dataset stratified by event


In [143]:
train_df, temp_df = train_test_split(
    contract_df,
    test_size=0.30,
    random_state=42,
    stratify=contract_df["event"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["event"],
)

In [144]:
id_cols = ["CONTRACT_ID", "borrower_id_mode"]
target_cols = ["time", "event"]

feature_cols = [c for c in contract_df.columns if c not in id_cols + target_cols]

X_train = train_df[feature_cols].copy()
X_val   = val_df[feature_cols].copy()
X_test  = test_df[feature_cols].copy()

ytime_train = train_df["time"].astype(float).to_numpy()
yevent_train = train_df["event"].astype(int).to_numpy()

ytime_val = val_df["time"].astype(float).to_numpy()
yevent_val = val_df["event"].astype(int).to_numpy()

ytime_test = test_df["time"].astype(float).to_numpy()
yevent_test = test_df["event"].astype(int).to_numpy()

# Scale numeric + OHE categorical

In [147]:
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
for c in cat_cols:
    X_train[c] = X_train[c].astype("string")
    X_val[c]   = X_val[c].astype("string")
    X_test[c]  = X_test[c].astype("string")

num_cols = [c for c in X_train.columns if c not in cat_cols]

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]), num_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train_mat = preprocess.fit_transform(X_train)
X_val_mat   = preprocess.transform(X_val)
X_test_mat  = preprocess.transform(X_test)

feature_names = preprocess.get_feature_names_out()

In [148]:
X_tr = X_train_mat.astype(np.float32)
X_va = X_val_mat.astype(np.float32)
X_te = X_test_mat.astype(np.float32)

t_tr = ytime_train.astype(np.float32)
t_va = ytime_val.astype(np.float32)
t_te = ytime_test.astype(np.float32)

e_tr = yevent_train.astype(np.int64)
e_va = yevent_val.astype(np.int64)
e_te = yevent_test.astype(np.int64)

# Save splits

In [162]:
# FULL feature matrices
pd.DataFrame(X_tr, columns=feature_names).assign(
    time=t_tr,
    event=e_tr
).to_csv("mnb_full_train_processed.csv", index=False)

pd.DataFrame(X_va, columns=feature_names).assign(
    time=t_va,
    event=e_va
).to_csv("mnb_full_val_processed.csv", index=False)

pd.DataFrame(X_te, columns=feature_names).assign(
    time=t_te,
    event=e_te
).to_csv("mnb_full_test_processed.csv", index=False)

# Check leakage

In [149]:
def auc_any_event(Xtr, yev_tr, Xva, yev_va):
    ytr = (yev_tr > 0).astype(int)
    yva = (yev_va > 0).astype(int)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xva)[:, 1]
    return roc_auc_score(yva, p), clf

auc, clf = auc_any_event(X_train_mat, yevent_train, X_val_mat, yevent_val)
print("AUC(any event vs censored) on val:", round(auc, 4))

# top coefficients to spot suspicious features
coefs = clf.coef_.ravel()
idx = np.argsort(np.abs(coefs))[::-1][:30]
top = pd.DataFrame({"feature": feature_names[idx], "coef": coefs[idx], "abs": np.abs(coefs[idx])}).sort_values("abs", ascending=False)
print(top.to_string(index=False))


AUC(any event vs censored) on val: 0.8862
                                     feature      coef      abs
            CONTRACT_FREQUENCY_TYPE_ad534644  2.404019 2.404019
             BORROWER_TYPE_OF_SETTLEMENT_0.0  2.265213 2.265213
               CONTRACT_LOAN_CONTRACT_TYPE_3 -2.115307 2.115307
CONTRACT_TYPE_OF_INTEREST_REPAYMENT_140002.0  2.063003 2.063003
               CONTRACT_LOAN_CONTRACT_TYPE_2 -1.933466 1.933466
               CONTRACT_LOAN_CONTRACT_TYPE_9  1.920092 1.920092
              CONTRACT_LOAN_CONTRACT_TYPE_11  1.767219 1.767219
              CONTRACT_LOAN_CONTRACT_TYPE_12  1.741541 1.741541
                 CONTRACT_LOAN_TYPE_5a06241e  1.662474 1.662474
         BORROWER_TYPE_OF_SETTLEMENT_missing  1.332312 1.332312
                 CONTRACT_LOAN_TYPE_7e2065f4  1.316365 1.316365
CONTRACT_TYPE_OF_INTEREST_REPAYMENT_110001.0  1.214094 1.214094
        CONTRACT_CREDIT_INTERMEDIARY_20001.0  1.206393 1.206393
                 CONTRACT_LOAN_TYPE_83910425  1.183173 1.18317

In [150]:
def auc_one_risk(Xtr, yev_tr, Xva, yev_va, risk=1):
    ytr = (yev_tr == risk).astype(int)
    yva = (yev_va == risk).astype(int)
    clf = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xva)[:, 1]
    return roc_auc_score(yva, p), clf

for r in [1, 2]:
    auc_r, _ = auc_one_risk(X_train_mat, yevent_train, X_val_mat, yevent_val, risk=r)
    print(f"AUC(risk {r} vs rest) on val:", round(auc_r, 4))


AUC(risk 1 vs rest) on val: 0.9857
AUC(risk 2 vs rest) on val: 0.8884


In [157]:
pd.crosstab(
    contract_df["event"],
    contract_df["CONTRACT_LOAN_CONTRACT_TYPE"],
    normalize="columns"
)

CONTRACT_LOAN_CONTRACT_TYPE,1,2,3,4,6,9,11,12
event,,,,,,,,
0,0.958808,0.992492,0.985122,0.93511,0.996914,0.884477,0.908443,0.956404
1,0.010841,0.004662,0.014849,0.06480,0.001052,0.002200,0.006176,0.001032
2,0.030352,0.002846,0.000029,0.00009,0.002034,0.113323,0.085381,0.042564


In [159]:
coef_df = pd.DataFrame({
    "feature": feature_names,
    "coef": np.abs(clf.coef_.ravel())
})

coef_df["base_feature"] = coef_df["feature"].str.rsplit("_", n=1).str[0]
coef_df.groupby("base_feature")["coef"].sum().sort_values(ascending=False).head(15)


,coef
base_feature,
CONTRACT_LOAN_CONTRACT_TYPE,10.549897
CONTRACT_LOAN_TYPE,9.107950
CONTRACT_TYPE_OF_INTEREST_REPAYMENT,7.411062
BORROWER_TYPE_OF_SETTLEMENT,6.651959
CONTRACT_FREQUENCY_TYPE,6.504470
CONTRACT_CREDIT_INTERMEDIARY,2.325238
BORROWER_COUNTY,1.989004
CONTRACT_REFINANCED,1.949220
BORROWER_CITIZENSHIP,1.197837


# Try if reduced feature set solve the high AUC problem


## Rerun the Scale + OHE part with reduced feature set

In [163]:
structural_features = [
    "CONTRACT_LOAN_CONTRACT_TYPE",
    "CONTRACT_LOAN_TYPE",
    "CONTRACT_TYPE_OF_INTEREST_REPAYMENT",
    "CONTRACT_FREQUENCY_TYPE",
]

reduced_feature_cols = [
    c for c in feature_cols if c not in structural_features
]

Xr_train = train_df[reduced_feature_cols].copy()
Xr_val   = val_df[reduced_feature_cols].copy()
Xr_test  = test_df[reduced_feature_cols].copy()

# categorical handling
cat_cols_r = Xr_train.select_dtypes(include=["object", "category"]).columns.tolist()
for c in cat_cols_r:
    Xr_train[c] = Xr_train[c].astype("string")
    Xr_val[c]   = Xr_val[c].astype("string")
    Xr_test[c]  = Xr_test[c].astype("string")

num_cols_r = [c for c in Xr_train.columns if c not in cat_cols_r]

In [164]:
preprocess_reduced = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ]), num_cols_r),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), cat_cols_r),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

Xr_tr = preprocess_reduced.fit_transform(Xr_train).astype(np.float32)
Xr_va = preprocess_reduced.transform(Xr_val).astype(np.float32)
Xr_te = preprocess_reduced.transform(Xr_test).astype(np.float32)

feature_names_r = preprocess_reduced.get_feature_names_out()

## Check if leakage persist

In [166]:
# Any event vs censored
clf_any = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf_any.fit(Xr_tr, (e_tr > 0).astype(int))
p_any = clf_any.predict_proba(Xr_va)[:, 1]
auc_any = roc_auc_score((e_va > 0).astype(int), p_any)
print("REDUCED AUC(any event vs censored) on val:", round(auc_any, 4))

# Risk 1 vs rest
clf_r1 = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf_r1.fit(Xr_tr, (e_tr == 1).astype(int))
p_r1 = clf_r1.predict_proba(Xr_va)[:, 1]
auc_r1 = roc_auc_score((e_va == 1).astype(int), p_r1)
print("REDUCED AUC(risk 1 vs rest) on val:", round(auc_r1, 4))

# Risk 2 vs rest
clf_r2 = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf_r2.fit(Xr_tr, (e_tr == 2).astype(int))
p_r2 = clf_r2.predict_proba(Xr_va)[:, 1]
auc_r2 = roc_auc_score((e_va == 2).astype(int), p_r2)
print("REDUCED AUC(risk 2 vs rest) on val:", round(auc_r2, 4))


REDUCED AUC(any event vs censored) on val: 0.7935
REDUCED AUC(risk 1 vs rest) on val: 0.9781
REDUCED AUC(risk 2 vs rest) on val: 0.795


In [167]:
clf = LogisticRegression(max_iter=2000, n_jobs=-1, class_weight="balanced")
clf.fit(Xr_tr, (e_tr == 1).astype(int))

coef_df = pd.DataFrame({
    "feature": feature_names_r,
    "coef": np.abs(clf.coef_.ravel())
})

coef_df["base_feature"] = coef_df["feature"].str.rsplit("_", n=1).str[0]
coef_df.groupby("base_feature")["coef"].sum().sort_values(ascending=False).head(10)


,coef
base_feature,
CONTRACT_CREDIT_INTERMEDIARY,6.713018
BORROWER_COUNTY,6.162502
CONTRACT_MATURITY,2.841024
BORROWER_CITIZENSHIP,2.754709
CONTRACT_CREDIT,2.700589
BORROWER_TYPE_OF_SETTLEMENT,1.993515
CONTRACT_LOAN,1.809500
CONTRACT_REFINANCED,1.392015
BORROWER_COUNTRY,1.199787
